# <center>Aggregation script<center>

#### import modules

In [ ]:
import pandas as pd
import numpy as np
import os
import math
from functools import reduce
from collections import defaultdict

#### load the files

In [ ]:
# list files without windows' shit that lists opened files separately
round_number = 26
input_file_names = [i for i in os.listdir("input/r{}".format(round_number)) if not "~$" in i]

# bear in mind that this name extraction works only if you follow the same conventions as I do
tool_name = [i for i in input_file_names if "tool" in i][0]
clean_name = [i for i in input_file_names if "final" in i][0]
# raw_name = [i for i in input_file_names if "raw" in i][0]

for fn in input_file_names:
    print(fn)

.ipynb_checkpoints
ISPA - aggregation rules - INDICATORS tool.xlsx
ISPA_2026_final_anonymized_data_Sep_2026_.xlsx


In [ ]:
# df_clean = pd.read_excel("input/r{}/{}".format(round_number, clean_name))
# df_raw = pd.read_excel("input/r{}/{}".format(round_number, raw_name))[["_uuid", "a1_4_settlement"]]
# data = df_clean.merge(df_raw, left_on='uuid', right_on='_uuid', how='left')

data = pd.read_excel("input/r{}/{}".format(round_number, clean_name))

In [ ]:
survey = pd.read_excel("input/r{}/{}".format(round_number, tool_name), sheet_name='survey')
choices = pd.read_excel("input/r{}/{}".format(round_number, tool_name), sheet_name='choices')

question_types = ['integer', 'select_one', 'select_multiple']
pattern = '|'.join(question_types)
survey_clean = survey[survey['type'].str.contains(pattern, na=False)].reset_index(drop=True)

In [ ]:
select_one_categorical = []
select_one_ordinal = []
select_multiple = []
numeric = []
select_multiple_options = {}
select_one_categorical_options = {}
select_one_ordinal_options_to_numbers = {}
select_one_ordinal_numbers_to_options = {}
select_one_yn = set()
unclear_type_cnt = 0
settlement_column = "A9_hromada_name" # "A8_raion"

questions_in_order = []

for _, row in survey_clean.iterrows():
    if row["Aggregation rules"] == "" or pd.isna(row["Aggregation rules"]):
        continue
    elif row["Aggregation rules"] == "set": # It's "take the value" in our kobo, but I want to set settlement manually
        settlement_column = row["name"]
        questions_in_order = [row["name"]] + questions_in_order
    elif row["type"] == "integer":
        numeric.append(row["name"])
        questions_in_order.append(row["name"])
    elif row["type"].startswith("select_multiple"):
        select_multiple.append(row["name"])
        questions_in_order.append(row["name"])

        options_list_name = row["type"].split(" ")[-1]
        options_list = choices[choices["list_name"] == options_list_name]["name"].to_list()
        select_multiple_options[row["name"]] = options_list
    elif row["type"].startswith("select_one") and row["Aggregation rules"] == "categorical":
        select_one_categorical.append(row["name"])
        questions_in_order.append(row["name"])

        options_list_name = row["type"].split(" ")[-1]
        options_list = choices[choices["list_name"] == options_list_name]["name"].to_list()
        select_one_categorical_options[row["name"]] = options_list

        if options_list_name == "y_n":
          select_one_yn.add(row["name"])
    elif row["type"].startswith("select_one") and row["Aggregation rules"] == "ordinal":
        select_one_ordinal.append(row["name"])
        questions_in_order.append(row["name"])

        options_list_name = row["type"].split(" ")[-1]
        options_list = choices[choices["list_name"] == options_list_name]
        o2n = {}
        n2o = {}
        for _, choice_row in options_list.iterrows():
          if choice_row["ordinal"] == "" or pd.isna(choice_row["ordinal"]):
            continue
          o2n[choice_row["name"]] = choice_row["ordinal"]
          n2o[choice_row["ordinal"]] = choice_row["name"]
        select_one_ordinal_options_to_numbers[row["name"]] = o2n
        select_one_ordinal_numbers_to_options[row["name"]] = n2o
    else:
        unclear_type_cnt += 1
        print(row["name"] + " is of unclear type, it won't be included in the output")


A7_oblast is of unclear type, it won't be included in the output
A8_raion is of unclear type, it won't be included in the output
A9_hromada is of unclear type, it won't be included in the output


#### Aggregate or not settlements based on KIs

In [ ]:
settlement_counts = data[settlement_column].value_counts()
valid_settlements = settlement_counts[settlement_counts >= 2].index

data = data[data[settlement_column].isin(valid_settlements)].copy()

#### select multiple aggregation

In [ ]:
select_multiple_columns = [sm + '/' + smo for sm in select_multiple for smo in select_multiple_options[sm]]
# this is a work-around because I did't have time to properly handle missing columns
smrm = set(['d8_1_education_barriers/no_school_in_the_settlement', 'd8_1_education_barriers/no_school_in_the_nearby_settlements', 'd8_1_education_barriers/schools_damaged', 'd8_1_education_barriers/not_enough_teachers', 'd8_1_education_barriers/not_enough_teaching_materials', 'd8_1_education_barriers/administrative_barriers', 'd8_1_education_barriers/physical_barriers', 'd8_1_education_barriers/security_barriers', 'd8_1_education_barriers/economic_barriers', 'd8_1_education_barriers/social_barriers', 'd8_1_education_barriers/other', 'd8_1_education_barriers/do_not_know', 'd8_1_education_barriers/prefer_not_to_answer'])
select_multiple_columns = [settlement_column] + [i for i in select_multiple_columns if i not in smrm]
select_multiple_data = data[select_multiple_columns]

In [ ]:
def aggregate_select_multiple(series):
    # remove NAs
    non_na = series.dropna()

    # if no non-NA values, return NA
    if len(non_na) == 0:
        return np.nan

    # if only one value, return it
    if len(non_na) == 1:
        return np.nan

    # if two values, see if there is a full consensus
    if len(non_na) == 2:
        if non_na.iloc[0] == non_na.iloc[1]:
            return non_na.iloc[0]
        else:
            return "NC"

    # if three or more values,
    # count occurrences of each value
    value_counts = non_na.value_counts()
    max_count = value_counts.max()
    total_count = len(non_na)

    # check if there's a majority (>= 50% or > 50% or y_n questions)
    original_q = series.name.split("/")[0] if isinstance(series.name, str) else None # for y_n
    if max_count / total_count < 0.5 or (original_q in select_one_yn and max_count / total_count == 0.5):
      return "NC"
    else:
      most_common_value = value_counts.idxmax()
      return most_common_value

select_multiple_aggregated = select_multiple_data.groupby(settlement_column).agg(aggregate_select_multiple).reset_index()

#### numeric aggregation

In [ ]:
numeric_columns = [settlement_column] + numeric
numeric_data = data[numeric_columns]

In [ ]:
def aggregate_numeric_func(series):
    # remove NAs
    data = series.dropna().tolist()
    if len(data) == 0:
      return np.nan

    return np.median(data)

def aggregate_numeric(df):
    group_col = df.columns[0]
    val_cols = df.columns[1:]

    results = {}

    for col in val_cols:
        results[f"{col}"] = df.groupby(group_col)[col].apply(aggregate_numeric_func)

    return pd.DataFrame(results).reset_index()

numeric_aggregated = aggregate_numeric(numeric_data)

#### select one categorical aggregation

In [ ]:
select_one_categorical_columns = [settlement_column] + select_one_categorical
select_one_categorical_data = data[select_one_categorical_columns]

In [ ]:
def select_one_explode(df):
    # keep the first column as it is
    first_col_name = df.columns[0]
    final_df = df[[first_col_name]].copy()

    # iterate through the rest of the columns
    for col in df.columns[1:]:
        # prefix_sep='/' gives you the 'col_name/value' format
        # dummy_na=False ensures we don't create a column for NAs
        dummies = pd.get_dummies(df[col], prefix=col, prefix_sep='/', dummy_na=False, dtype=float)
        # stitch everything together
        final_df = pd.concat([final_df, dummies], axis=1)

    return final_df

In [ ]:
select_one_categorical_data_exploded = select_one_explode(select_one_categorical_data)
select_one_categorical_aggregated = select_one_categorical_data_exploded.groupby(settlement_column).agg(aggregate_select_multiple).reset_index()

#### select one ordinal aggregation

In [ ]:
select_one_ordinal_columns = [settlement_column] + select_one_ordinal
select_one_ordinal_data = data[select_one_ordinal_columns]

In [ ]:
def map_values(df, mapping_dict):
    # Create a copy to avoid modifying the original dataframe
    new_df = df.copy()

    # get all columns except the first one
    # if you have specific columns in the mapping_dict,
    # it's safer to iterate through the dictionary keys
    for col, mapper in mapping_dict.items():
        if col in new_df.columns:
            # .map() replaces values based on the dictionary.
            # Values not in the dictionary will become NaN.
            new_df[col] = new_df[col].map(mapper)

    return new_df

# Usage:
# df_mapped = map_text_to_numbers(df, your_mapping_dictionary)

In [ ]:
select_one_ordinal_data_mapped = map_values(select_one_ordinal_data, select_one_ordinal_options_to_numbers)

In [ ]:
def aggregate_ordinal_numeric_func(series):
    # remove NAs
    data = series.dropna().tolist()
    n = len(data)

    # if there's just one value, aggregate to NA
    if n == 1:
        return np.nan

    # if there are two values
    if n == 2:
        if data[0] == data[1]:
            return data[0]
        else:
            return "NC"

    # for 3 or more values
    if n >= 3:
        return math.ceil(np.median(data))

    return np.nan

def aggregate_ordinal_numeric(df):
    group_col = df.columns[0]
    val_cols = df.columns[1:]

    results = {}

    for col in val_cols:
        results[f"{col}"] = df.groupby(group_col)[col].apply(aggregate_ordinal_numeric_func)

    return pd.DataFrame(results).reset_index()

In [ ]:
select_one_ordinal_data_mapped_aggregated = aggregate_ordinal_numeric(select_one_ordinal_data_mapped)

In [ ]:
def map_values_back(df, mapping_dict):
    new_df = df.copy()

    for col, mapper in mapping_dict.items():
        target_col = col

        if target_col in new_df.columns:
            # Use .apply() with a lambda to handle "NC" and rounding safely
            # This avoids the Reindexing error caused by .map() on mixed types
            new_df[target_col] = new_df[target_col].apply(
                lambda x: mapper.get(int(round(float(x))), x)
                if isinstance(x, (int, float)) and pd.notna(x)
                else x
            )

    return new_df

In [ ]:
select_one_ordinal_aggregated = map_values_back(select_one_ordinal_data_mapped_aggregated, select_one_ordinal_numbers_to_options)

#### stitch together and save

In [ ]:
dfs = [
    select_multiple_aggregated,
    numeric_aggregated,
    select_one_categorical_aggregated,
    select_one_ordinal_aggregated
]

# we use 'reduce' to merge all dataframes one after another
# on the first column name
first_col = dfs[0].columns[0]
df_final = reduce(lambda left, right: pd.merge(left, right, on=first_col, how='outer'), dfs)

In [ ]:
# add text select multiple columns
groups = defaultdict(list)
for col in df_final.columns:
    if '/' in col:
        prefix, option = col.split('/', 1)
        groups[prefix].append((col, option))

# For each group, create a new column with space-joined options where value == 1
for prefix, members in groups.items():
    def join_options(row):
        return ' '.join(option for col, option in members if row[col] == 1)
    df_final[prefix] = df_final.apply(join_options, axis=1)

In [ ]:
df_final.to_excel("output/r{}/r{}_dataset_aggregated.xlsx".format(round_number, round_number), index=False)

#### ad-hocs

In [ ]:
# don't call
output = "output/r26/r26_dataset_aggregated.xlsx"
df = pd.read_excel(output)

sampl = pd.read_excel("input/r26/sampling_r26.xlsx", sheet_name="Sampling frame")[["A8_raion", "A9_hromada"]]
df = df.merge(sampl, on='a1_4_settlement', how='left')


df.to_excel("output/r25/gca_r25_dataset_aggregated.xlsx", sheet_name="main", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'input/r26/sampling_r26.xlsx'

### <center>Created by Andrei Andrievsky<center>
#### Editer for r26 by Kostiantyn Solovei